# Day 3 (Re-run) — Modeling on Expanded Dataset
YouTube Real-Time Engagement Prediction Project

Goal for today:
1. Load Day 2's EXPANDED training table (537 videos, 14 categories)
2. Prepare features (encode category, drop non-feature columns)
3. Evaluate with 5-fold cross-validation
4. Train a final model on all data + do one train/test split just to
   produce a clean "predicted vs actual" plot for the report
5. Look at feature importance — now with real category variety
6. Save as a VERSIONED model (v2) — the original model is kept untouched
   for direct before/after comparison

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import os

sns.set_style("whitegrid")

TRAINING_TABLE_PATH = "../data/processed/training_table_v2.csv"
MODEL_PATH = "../models/engagement_model_v2.pkl"
FEATURE_LIST_PATH = "../models/feature_columns_v2.pkl"

df = pd.read_csv(TRAINING_TABLE_PATH)
print("Shape:", df.shape)
df.head()

## 1. Prepare features and target

Dropped columns:
- `video_id`: identifier, not a feature
- `target_snapshot_age_hours`: metadata about WHEN the target was measured
  (18-30h), not something known at prediction time in the same sense as
  the early-window features — excluded to keep the feature set honest

`category_id` is a NOMINAL category (Tech, Music, Gaming...), not ordinal,
so it's one-hot encoded rather than used as a raw number (a raw number
would wrongly imply category 20 is "more" than category 10).

NEW (v2): With 14 categories now (vs. 6 originally), several have very
few samples (e.g. Travel & Events might have only 3). One-hot encoding
every category separately would create sparse, noisy columns the model
can't learn anything reliable from. Using Day 2's own category counts,
categories below a minimum sample threshold are grouped into "Other"
rather than each getting their own column — this uses information already
computed in Day 2 to make Day 3's encoding smarter.

In [ ]:
MIN_SAMPLES_PER_CATEGORY = 10  # categories with fewer videos than this get grouped into "Other"

category_counts = df["category_id"].value_counts()
print("Category counts (from this training table):")
print(category_counts)

rare_categories = category_counts[category_counts < MIN_SAMPLES_PER_CATEGORY].index.tolist()
print(f"\nCategories grouped into 'Other' (< {MIN_SAMPLES_PER_CATEGORY} samples): {rare_categories}")

df["category_grouped"] = df["category_id"].apply(
    lambda c: "Other" if c in rare_categories else str(c)
)

In [ ]:
feature_cols_raw = [
    "category_grouped", "duration_seconds", "upload_hour", "upload_dayofweek",
    "is_weekend", "early_snapshot_age_hours", "early_snapshot_count",
    "early_views", "early_likes", "early_comments", "early_views_per_hour",
    "early_like_rate", "early_comment_rate", "early_engagement_rate",
]

X = df[feature_cols_raw].copy()
X = pd.get_dummies(X, columns=["category_grouped"], prefix="cat")
y = df["target_engagement_rate"].copy()

print("Final feature matrix shape:", X.shape)
print("Feature columns:", list(X.columns))

## 2. Hyperparameter tuning
With 537 samples (vs. 162 originally), there's now enough data to justify
tuning instead of using fixed defaults. A small grid search over the most
impactful Random Forest parameters, evaluated with the same 5-fold CV.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [200, 300, 500],
    "max_depth": [4, 6, 8, None],
    "min_samples_leaf": [1, 3, 5],
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=kf,
    scoring="r2",
    n_jobs=-1,
)
grid_search.fit(X, y)

print("Best parameters found:", grid_search.best_params_)
print("Best CV R2:", round(grid_search.best_score_, 3))

best_params = grid_search.best_params_

## 3. Cross-validated evaluation (5-fold) — using tuned parameters
With more samples now, a single 80/20 split is more trustworthy than
before, but we still report 5-fold CV as the primary metric for
consistency with the original (v1) evaluation.

In [ ]:
model = RandomForestRegressor(random_state=42, **best_params)

rmse_scores = -cross_val_score(model, X, y, cv=kf, scoring="neg_root_mean_squared_error")
mae_scores = -cross_val_score(model, X, y, cv=kf, scoring="neg_mean_absolute_error")
r2_scores = cross_val_score(model, X, y, cv=kf, scoring="r2")

print("5-Fold Cross-Validation Results:")
print(f"RMSE: {rmse_scores.mean():.3f} (+/- {rmse_scores.std():.3f})")
print(f"MAE:  {mae_scores.mean():.3f} (+/- {mae_scores.std():.3f})")
print(f"R2:   {r2_scores.mean():.3f} (+/- {r2_scores.std():.3f})")

## 3. Train/test split — for a clean "predicted vs actual" plot
This is just for visualization purposes; the CV scores above are the
numbers to actually report/trust given the small sample size.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model_demo = RandomForestRegressor(random_state=42, **best_params)
model_demo.fit(X_train, y_train)
y_pred = model_demo.predict(X_test)

test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("Test set size:", len(X_test))
print(f"Test RMSE: {test_rmse:.3f}")
print(f"Test MAE:  {mean_absolute_error(y_test, y_pred):.3f}")
print(f"Test R2:   {r2_score(y_test, y_pred):.3f}")

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.7)
lims = [0, max(y_test.max(), y_pred.max()) * 1.05]
plt.plot(lims, lims, "r--", label="Perfect prediction")
plt.xlabel("Actual Engagement Rate (%) at ~24h")
plt.ylabel("Predicted Engagement Rate (%)")
plt.title("Predicted vs Actual Engagement Rate")
plt.legend()
plt.show()

## 4. Feature importance
Shows which early signals the model relies on most to predict future
engagement — good for explaining WHY a prediction was made, not just WHAT
it predicted.

In [ ]:
final_model = RandomForestRegressor(random_state=42, **best_params)
final_model.fit(X, y)

importances = pd.Series(final_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
importances.plot(kind="barh")
plt.gca().invert_yaxis()
plt.title("Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

print(importances)

## 5. Save the final model (trained on ALL 537 videos) — versioned as v2
The ORIGINAL model (engagement_model.pkl, trained on 162 videos) is left
untouched, so both versions exist side by side for comparison and so the
dashboard/live pipeline isn't disrupted until we deliberately swap it.

In [ ]:
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
joblib.dump(final_model, MODEL_PATH)
joblib.dump(list(X.columns), FEATURE_LIST_PATH)

print(f"Saved model to {MODEL_PATH}")
print(f"Saved feature column list to {FEATURE_LIST_PATH}")

## 6. Before / After Comparison
Original model results (162 videos, 6 categories), recorded from Day 3's
first run, compared directly against this new model's cross-validated
results.

In [ ]:
original_results = {
    "Dataset size": 162,
    "Categories": 6,
    "R2 (mean)": 0.835,
    "R2 (std)": 0.135,
    "RMSE": 1.15,
    "MAE": 0.64,
}

new_results = {
    "Dataset size": len(df),
    "Categories": df["category_id"].nunique(),
    "R2 (mean)": round(r2_scores.mean(), 3),
    "R2 (std)": round(r2_scores.std(), 3),
    "RMSE": round(rmse_scores.mean(), 3),
    "MAE": round(mae_scores.mean(), 3),
}

comparison = pd.DataFrame([original_results, new_results], index=["Original (v1)", "Expanded (v2)"])
print(comparison)